<a href="https://colab.research.google.com/github/12sandra/Data-Science-Projects/blob/main/insurance_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

importing librraies

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import(train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV)
from sklearn.preprocessing import StandardScaler

#models
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
#pickle
import joblib

importing dataset

In [ ]:
file_path="/content/insurance.csv"
df=pd.read_csv(file_path)

###EDA

In [ ]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [ ]:
df.shape

(1338, 7)

In [ ]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [ ]:
df.nunique()

,0
age,47
sex,2
bmi,548
children,6
smoker,2
region,4
charges,1337


In [ ]:
df.isnull().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
charges,0


In [ ]:
df.duplicated().sum()

np.int64(1)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
numeric_df=df.select_dtypes(include=['float64','int64'])
categorical_df=df.select_dtypes(include=['object'])

In [ ]:
q1=numeric_df.quantile(.25)
q3=numeric_df.quantile(.75)
iqr=q3-q1

df[numeric_df.columns]=numeric_df.clip(
    lower=q1-(1.5*iqr),
    upper=q3+(1.5*iqr),axis=1
)


one-hot Encoding

In [ ]:
df=pd.get_dummies(df,columns=categorical_df.columns,drop_first=True,dtype=int)

In [ ]:
numeric_df=numeric_df.drop('charges',axis=1)

In [ ]:
X=df.drop('charges',axis=1)
y=df['charges']

In [ ]:

columns = X.columns

import joblib
joblib.dump(columns, "columns.pkl")

['columns.pkl']

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42)

In [ ]:
sscaler=StandardScaler()

In [ ]:
X_train = sscaler.fit_transform(X_train)  # learn from train
X_test = sscAaler.transform(X_test)


Scaling

In [ ]:
model_lr=LinearRegression()
model_lr.fit(X_train,y_train)
y_pred_lr=model_lr.predict(X_test)
y_train.head()

,charges
763,3070.8087
1079,15161.5344
178,8823.2790
287,14256.1928
1290,7133.9025


In [ ]:
#evaluate
print("R2 Score:", r2_score(y_test, y_pred_lr))
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("MSE:", mean_squared_error(y_test, y_pred_lr))


R2 Score: 0.8015687812406103
MAE: 3107.712323168449
MSE: 22039423.93752282


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [ ]:
#evaluate
print("R2 Score:", r2_score(y_test, rf_pred))
print("MAE:", mean_absolute_error(y_test, rf_pred))
print("MSE:", mean_squared_error(y_test, rf_pred))


R2 Score: 0.8263177125514868
MAE: 2331.6413334583626
MSE: 19290601.486240923


In [ ]:
rf.feature_importances_

array([0.16344645, 0.17495909, 0.02882184, 0.00926836, 0.60166382,
       0.00803788, 0.00808542, 0.00571713])

In [ ]:
cv_scores=cross_val_score(rf,X,y,cv=5)
print("cv_accuracy : ",cv_scores)
print("mean cv score : ",cv_scores.mean())

cv_accuracy :  [0.81149982 0.70714427 0.82892458 0.78332892 0.82186538]
mean cv score :  0.7905525954605629


In [ ]:
param_grid={
    'n_estimators':[100,200,500],
    'max_depth':[3,5,10,20]
}

grid=GridSearchCV(
    rf,param_grid=param_grid,cv=5
)
grid.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid.best_params_)

Best Parameters: {'max_depth': 3, 'n_estimators': 500}


In [ ]:
print("Best Score:", grid.best_score_)

Best Score: 0.8053898916073738


In [ ]:
best_rf = grid.best_estimator_

y_pred = best_rf.predict(X_test)

In [ ]:
print("Test Accuracy:", r2_score(y_test, y_pred))

Test Accuracy: 0.8548927889665341


In [ ]:
joblib.dump(sscaler,"scaler.pkl")
joblib.dump(best_rf,"model.pkl")

['model.pkl']